In [1]:
#import torch
#from unsloth import FastLanguageModel
import polars as pl
import csv
from tqdm import tqdm
#from sentence_transformers import SentenceTransformer
from llms import legal_rag_agent

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# ==========================================
# 1. SETUP THE LIBRARIAN (RoBERTa + FAISS)
# ==========================================
print(" Loading the Librarian (RoBERTa)...")
DATASET_PATH = "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv"
SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"

# Load the data to get the unique citations for FAISS
roberta = roberta() # Assuming your class is named 'roberta' based on your previous code
full_df = roberta.load_datasets([DATASET_PATH])
unique_cits = (
    full_df.select(pl.col("parsed_citations").list.explode())
    .drop_nulls().unique().to_series().to_list()
)

# Load Fine-Tuned RoBERTa
finetuned_model = roberta.load_model(SSD_MODEL_PATH)
finetuned_model.max_seq_length = 512
retriever = roberta_legal_retriever(model=finetuned_model, unique_citations=unique_cits)

# ==========================================
# 2. SETUP THE SPOKESPERSON (Mistral)
# ==========================================
print("\n Loading the Spokesperson (Mistral-7B)...")
max_seq_length = 2048
dtype = None 
load_in_4bit = True 

mistral = mistral() 
mistral_model, mistral_tokenizer = mistral.load_model_and_tokenizer(max_seq_length = max_seq_length, 
            dtype = dtype, load_in_4bit = load_in_4bit, model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit")

# mistral_model, mistral_tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", # 4-bit quantized to fit in your VRAM
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
# )
# FastLanguageModel.for_inference(mistral_model) # Enable Unsloth 2x faster inference

# ==========================================
# 3. THE STRICT PARALEGAL PROMPT
# ==========================================
rag_prompt_template = """Du bist ein hochprofessioneller österreichischer Rechtsassistent.
Ein Retrieval-System hat den folgenden Sachverhalt analysiert und die exakten relevanten Gesetzesstellen gefunden.

Deine Aufgabe:
1. Lies den Sachverhalt und die gefundenen Paragraphen.
2. Schreibe eine kurze, höfliche und professionelle Antwort an den Nutzer.
3. Nenne in deiner Antwort NUR die bereitgestellten Paragraphen.
4. WICHTIG: Erfinde KEINE Gesetzestexte. Erkläre den Inhalt der Paragraphen NICHT, da du sonst halluzinieren könntest. Nenne ausschließlich ihre Namen als Referenz für den Nutzer.

Gefundene relevante Paragraphen:
{citations}

Sachverhalt des Nutzers:
{query}

### Antwort:
"""

# ==========================================
# 4. THE LIVE RAG PIPELINE
# ==========================================
def ask_rag_agent(user_query, k=3):
    print("\n" + "="*50)
    print(" RAG AGENT PROCESSING...")
    print("="*50)
    
    # Step A: Librarian retrieves citations
    print("1. Searching Austrian Law Database...")
    results = retriever.retrieve(user_query, k=k)
    retrieved_citations = [r["citation"] for r in results]
    citations_string = "\n".join([f"- {cit}" for cit in retrieved_citations])
    
    # Step B: Format the prompt for Mistral
    print("2. Formulating response...")
    prompt = rag_prompt_template.format(citations=citations_string, query=user_query)
    inputs = mistral_tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Step C: Mistral generates the answer
    outputs = mistral_model.generate(
        **inputs, 
        max_new_tokens=150, # Keep it concise
        use_cache=True,
        pad_token_id=mistral_tokenizer.eos_token_id 
    )
    
    # Step D: Decode and print
    response = mistral_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    final_answer = response.split("### Antwort:\n")[-1].strip()
    
    print("\n AGENT RESPONSE:")
    print("-" * 50)
    print(final_answer)
    print("-" * 50)


# Let's test it!
test_case = "Ein selbstständiger IT-Berater kauft einen neuen Laptop für 2.500 Euro, den er zu 80% beruflich und zu 20% privat nutzt. Er möchte wissen, wie er diesen im Rahmen der Einkommensteuererklärung als Betriebsausgabe absetzen kann."

ask_rag_agent(test_case)

 Loading the Librarian (RoBERTa)...
Setup complete. Caching and Output mapped to SSD: {ssd_base_path}
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv...
Successfully loaded and merged 1 datasets.
Total training cases available: 1587
Loading model: /mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Building FAISS Index (Citation Strings Only) ===
Encoding 2040 unique citations...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Encoding took 0.86 seconds
Index built with 2040 vectors.

 Loading the Spokesperson (Mistral-7B)...
Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit...
==((====))==  Unsloth 2026.4.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.885 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 RAG AGENT PROCESSING...
1. Searching Austrian Law Database...
2. Formulating response...


/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) 


 AGENT RESPONSE:
--------------------------------------------------
Liebe Person,

Ich habe den Sachverhalt analysiert und die relevanten Gesetzesstellen gefunden.

Für die Berücksichtigung von Ausgaben als Betriebsausgaben bei der Einkommensteuererklärung gelten die Bestimmungen des DurchschnittssatzV Werbungskosten 2001.

Zudem ist die Bestimmung des EStG 1988 §20 Abs1 relevant, welche die Voraussetzungen für die Berücksichtigung von Ausgaben als Betriebsausgaben festlegt.

Ich empfehle,
--------------------------------------------------


In [3]:
# Let's test it!
test_case = "Mein Arbeitgeber hat mit mein Lohn nicht ausbezahlt, obwohl ich meine Arbeit ordnungsgemäß erledigt habe. Welche rechtlichen Schritte kann ich unternehmen, um meinen Lohn einzufordern?"

ask_rag_agent(test_case)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 RAG AGENT PROCESSING...
1. Searching Austrian Law Database...
2. Formulating response...

 AGENT RESPONSE:
--------------------------------------------------
Liebe Person,

Ich habe Ihren Sachverhalt analysiert und die folgenden Paragraphen gefunden, die Ihnen möglicherweise hilfreich sein könnten:

- EStG 1988 §82: Dieser Paragraph regelt die Verpflichtung des Arbeitgebers, den Lohn zu bezahlen.
- HGB §4 Abs1: Dieser Paragraph regelt die Verpflichtung des Arbeitgebers, die Lohnsumme dem Arbeitnehmer rechtzeitig und ordnungsgemäß zu melden.
- HBG §17: Dieser Paragraph regelt
--------------------------------------------------


In [2]:
# Make sure to import LegalRAGAgent from your llms.py file
# from llms import LegalRAGAgent

if __name__ == "__main__":
    DATASET_PATH = ["/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv"]
    SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"

    # 1. Initialize the Agent with your paths
    rag = legal_rag_agent(datasets_paths=DATASET_PATH, roberta_ft_model_path=SSD_MODEL_PATH)
    
    # 2. Load the heavy models into VRAM (Only happens once!)
    rag.prepare_agent()
    
    # 3. Now you can ask as many questions as you want instantly!
    query_1 = "Ein IT-Berater kauft einen Laptop für 2.500 Euro (80% beruflich, 20% privat). Wie kann er das absetzen?"
    rag.ask(query_1, k=3)
    
    query_2 = "Ich habe Spenden an eine anerkannte Hilfsorganisation getätigt. Kann ich das von der Steuer absetzen?"
    rag.ask(query_2, k=2)

 PREPARING RAG AGENT (This will take a minute...)
 1/2 Loading RoBERTa Retriever...
Setup complete. Caching and Output mapped to SSD: {ssd_base_path}
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv...
Successfully loaded and merged 1 datasets.
Total training cases available: 1587
Loading model: /mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Building FAISS Index (Citation Strings Only) ===
Encoding 2040 unique citations...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Encoding took 0.91 seconds
Index built with 2040 vectors.

 2/2 Loading Mistral-7B Generator...
Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit...
==((====))==  Unsloth 2026.4.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.885 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 RAG AGENT IS FULLY LOADED AND READY FOR QUERIES!

 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['ABGB §1053', 'ABGB §1090', '62008CJ0538 X Holding VORAB']
   -> FOUND_citations_string: - ABGB §1053
- ABGB §1090
- 62008CJ0538 X Holding VORAB
 2. Mistral is formulating the response...


/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) 


 AGENT RESPONSE:
--------------------------------------------------
Lieber Nutzer,

Ich habe Ihren Sachverhalt analysiert und habe die exakten relevanten Gesetzesstellen gefunden. Hier eine Übersicht:

- ABGB §1053
- ABGB §1090
- 62008CJ0538 X Holding VORAB

Ich wünsche Ihnen viel Erfolg bei Ihrer Steuerung.

Mit freundlichen Grüßen,
Ihr Rechtsassistent
--------------------------------------------------

 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['PSG 1993 §18', 'EntwicklungshilfeG 1974']
   -> FOUND_citations_string: - PSG 1993 §18
- EntwicklungshilfeG 1974
 2. Mistral is formulating the response...

 AGENT RESPONSE:
--------------------------------------------------
Lieber Nutzer,

Ich habe Ihre Frage betreffend die Steuererleichterung bei Spenden an eine anerkannte Hilfsorganisation untersucht. Hier eine Übersicht der relevanten Gesetzesstellen:

- PSG 1993 §18
- EntwicklungshilfeG 1974

Ich

In [3]:
# 3. Now you can ask as many questions as you want instantly!
query_3 = "Ich habe mein Lohn von meinem Arbeitgeber nicht erhalten, obwohl ich meine Arbeit ordnungsgemäß erledigt habe. Welche rechtlichen Schritte kann ich unternehmen, um meinen Lohn einzufordern?"
rag.ask(query_3, k=3)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['EStG 1988 §82', 'HGB §4 Abs1', 'HGB §171']
   -> FOUND_citations_string: - EStG 1988 §82
- HGB §4 Abs1
- HGB §171
 2. Mistral is formulating the response...


/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



 AGENT RESPONSE:
--------------------------------------------------
Lieber Nutzer,

Ich habe Ihren Fall analysiert und die exakten relevanten Gesetzesstellen gefunden. Hier eine Übersicht:

- EStG 1988 §82: Regelt die Verpflichtung des Arbeitgebers, den Lohn zu zahlen.
- HGB §4 Abs1: Regelt die Verpflichtung des Arbeitgebers, die Lohnsumme dem Arbeitnehmer zu zeigen.
- HGB §171: Regelt die Verpflichtung des Arbeitgebers, die Lohnsumme dem Arbeitnehmer rechtzeitig zu zahlen.

Ich wünsche Ihnen viel Erfolg bei der Lösung Ihres Problems.

Mit freundlichen Grüßen,
Ihr Rechtsassistent
--------------------------------------------------


'Lieber Nutzer,\n\nIch habe Ihren Fall analysiert und die exakten relevanten Gesetzesstellen gefunden. Hier eine Übersicht:\n\n- EStG 1988 §82: Regelt die Verpflichtung des Arbeitgebers, den Lohn zu zahlen.\n- HGB §4 Abs1: Regelt die Verpflichtung des Arbeitgebers, die Lohnsumme dem Arbeitnehmer zu zeigen.\n- HGB §171: Regelt die Verpflichtung des Arbeitgebers, die Lohnsumme dem Arbeitnehmer rechtzeitig zu zahlen.\n\nIch wünsche Ihnen viel Erfolg bei der Lösung Ihres Problems.\n\nMit freundlichen Grüßen,\nIhr Rechtsassistent'

In [5]:
rag.ask("Ich bin Studentin in Österreich, wo ist meine Geringfügigkeitsgrenze?", k=5)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['DBAbk Ungarn 1976 Art22 Abs2', 'NAG 2005 §2 Abs4 Z3', 'DBAbk Schweiz 1975 Art15 Abs1', 'DBAbk Ungarn 1976 Art22', 'DBAbk Liechtenstein 1971 Art14']
   -> FOUND_citations_string: - DBAbk Ungarn 1976 Art22 Abs2
- NAG 2005 §2 Abs4 Z3
- DBAbk Schweiz 1975 Art15 Abs1
- DBAbk Ungarn 1976 Art22
- DBAbk Liechtenstein 1971 Art14
 2. Mistral is formulating the response...


/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/sj5/.micromamba/envs/llm_course/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



 AGENT RESPONSE:
--------------------------------------------------
Liebe Studentin,

Ich habe Ihre Frage analysiert und habe die folgenden Paragraphen gefunden, die möglicherweise Ihre Frage beantworten könnten:

- DBAbk Ungarn 1976 Art22 Abs2
- NAG 2005 §2 Abs4 Z3
- DBAbk Schweiz 1975 Art15 Abs1
- DBAbk Ungarn 1976 Art22
- DBAbk Liechtenstein 1971 Art14

Ich wünsche Ihnen viel Erfolg bei der Suche nach Ihrer Antwort.

Mit freundlichen Grüßen,
Ihr Assistent
--------------------------------------------------


'Liebe Studentin,\n\nIch habe Ihre Frage analysiert und habe die folgenden Paragraphen gefunden, die möglicherweise Ihre Frage beantworten könnten:\n\n- DBAbk Ungarn 1976 Art22 Abs2\n- NAG 2005 §2 Abs4 Z3\n- DBAbk Schweiz 1975 Art15 Abs1\n- DBAbk Ungarn 1976 Art22\n- DBAbk Liechtenstein 1971 Art14\n\nIch wünsche Ihnen viel Erfolg bei der Suche nach Ihrer Antwort.\n\nMit freundlichen Grüßen,\nIhr Assistent'

In [2]:
if __name__ == "__main__":
    # Your Paths
    TRAIN_DATASET_PATH = [
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv", # Assuming this is your KStG file
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv"   # Assuming this is your BAO file
    ]
    SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"
    
    # Professor's Paths
    PROJECT_INPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/dataset_clean.csv"
    PROJECT_OUTPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/model_output_rag_3.csv"

    # 1. Initialize and Load
    rag = legal_rag_agent(datasets_paths=TRAIN_DATASET_PATH, roberta_ft_model_path=SSD_MODEL_PATH)
    rag.prepare_agent()
    
    # 2. Load the project Dataset
    print(f"\n Loading project dataset from: {PROJECT_INPUT_CSV}")
    project_df = pl.read_csv(PROJECT_INPUT_CSV)

    project_df=project_df.head(10)
    
    # 3. Process the dataset and save the output
    print(f" Running Batch Inference on {project_df.height} questions in FULL TEXT mode...")
    
    with open(PROJECT_OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Write the exact columns the project wants: id, answer
        writer.writerow(["id", "answer"])
        
        # Loop through every row in the project's dataset
        for row in tqdm(project_df.iter_rows(named=True), total=project_df.height):
            q_id = row.get("id", "")
            q_prompt = row.get("prompt", "")
            
            # The Magic: Ask the RAG agent in "full_text" mode!
            # We use k=5 to give Mistral enough context for complex questions
            answer = rag.ask(user_query=q_prompt, k=5, max_tokens=300, mode="full_text")
            
            # Save the result to the CSV
            writer.writerow([q_id, answer])
            
    print(f"\n All done! The final answers are saved at: {PROJECT_OUTPUT_CSV}")

 PREPARING RAG AGENT (This will take a minute...)
 1/2 Loading RoBERTa Retriever...
Setup complete. Caching and Output mapped to SSD: {ssd_base_path}
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv...
Successfully loaded and merged 3 datasets.
Total training cases available: 5066
Loading model: /mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Building FAISS Index (Citation Strings Only) ===
Encoding 5438 unique citations...


Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Encoding took 1.98 seconds
Index built with 5438 vectors.

 2/2 Loading Mistral-7B Generator...
Loading unsloth/mistral-7b-instruct-v0.3-bnb-4bit...
==((====))==  Unsloth 2026.4.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.885 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...


 AGENT RESPONSE:
--------------------------------------------------
Beim Forderungsverzicht eines Gesellschafters gilt für die Körperschaft eine Verlusteintragung nach § 23 UmgrStG 1991 (§ 23 Abs 1 UmgrStG 1991).

### Zusätzliche Informationen:

Dieser Verlust kann jedoch nach § 27 UmgrStG 1991 (§ 27 UmgrStG 1991) nur dann als steuerlich relevant angesehen werden, wenn die Körperschaft keinen Ersatz für die Forderung erzielt hat.

### Zusätzliche Informationen:

Die Körperschaft muss die Verlusteintragung nach § 23 Abs 1 UmgrStG 1991 (§ 23 Abs 1 UmgrStG 1991) innerhalb des Jahres nach dem Verzicht durchführen.

### Zusätzliche Informationen:

Die Verlusteintragung nach § 23 Abs 1 UmgrStG 1991 (§ 23 Abs 1 UmgrStG 1991) kann jedoch nach § 1 UmgrStG 1991 (§ 1
--------------------------------------------------

 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['DBAbk Schweiz 1975 Art23 Abs2', 'DBAbk Schwe

 90%|█████████ | 9/10 [00:58<00:07,  7.10s/it]Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 AGENT RESPONSE:
--------------------------------------------------
Die Verlustverrechnung ausländischer Gruppenmitglieder ist von der inländischer gruppenmitglieder unterschiedlich, da sie von den jeweiligen nationalen Steuergesetzen bestimmt wird.

Beispiel:
- Schweiz: DBAbk Schweiz 1975 Art23 Abs2, Art15 Abs1, Art4
- Italien: DBAbk Italien 1985 Art23 Abs3 lita

### Zitierung:

(DBAbk Schweiz 1975 Art23 Abs2, Art15 Abs1, Art4; DBAbk Italien 1985 Art23 Abs3 lita)
--------------------------------------------------

 RAG AGENT PROCESSING...
 1. RoBERTa is searching the FAISS Index- searching Austrian Law Database...
   ↳ FOUND_retrieved_citations: ['GSpG 1989 §28 Abs1', 'AktG 1965 §15', 'GSpG 1989 §28 Abs2', 'AktG 1965 §174 Abs4', 'AktG 1965 §174 Abs3']
 2. Mistral is formulating the response...


100%|██████████| 10/10 [01:07<00:00,  6.73s/it]


 AGENT RESPONSE:
--------------------------------------------------
Ja, Steuerumlagen zwischen Gruppenträgern und Gruppenmitgliedern sind steuerpflichtig, wenn sie nicht im Rahmen der Gruppensteuerung (§ 28 Abs 1 GSpG 1989) oder als Vergütung für Leistungen (§ 15 AktG 1965) erfolgen.

### Zusätzliche Informationen:

- Steuerumlagen zwischen Gruppenträgern und Gruppenmitgliedern können unter bestimmten Bedingungen steuerbefreit sein, wenn sie im Rahmen der Gruppensteuerung (§ 28 Abs 2 GSpG 1989) erfolgen.
- Steuerumlagen können auch als Vergütung für Leistungen anerkannt werden, wenn sie den Anforderungen des § 174 AktG 1965 entsprechen.
- Steuerumlagen können auch als Vergütung für Leistungen anerkannt werden, wenn sie den Anforderungen des § 174 Abs 3 AktG 1965 entsprechen.
--------------------------------------------------

 All done! The final answers are saved at: /mnt/red/red_hanka_bcthesis/llm/model_output_rag_3.csv


In [ ]:
#testing of rag on dataset_clean - project
if __name__ == "__main__":
    # Your Paths
    TRAIN_DATASET_PATH = [
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv", 
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_ASVGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_DBAdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_GrEStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv"
    ]
    SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"
    
    # Professor's Paths
    PROJECT_INPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/dataset_clean.csv"
    PROJECT_OUTPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/model_output_rag_4.csv"

    # 1. Initialize and Load
    rag = legal_rag_agent(datasets_paths=TRAIN_DATASET_PATH, roberta_ft_model_path=SSD_MODEL_PATH)
    rag.prepare_agent()
    
    # 2. Load the project Dataset
    print(f"\n Loading project dataset from: {PROJECT_INPUT_CSV}")
    project_df = pl.read_csv(PROJECT_INPUT_CSV)

    project_df=project_df.head(5)
    
    # 3. Process the dataset and save the output
    print(f" Running Batch Inference on {project_df.height} questions in FULL TEXT mode...")
    
    with open(PROJECT_OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Write the exact columns the project wants: id, answer
        writer.writerow(["id", "answer"])
        
        # Loop through every row in the project's dataset
        for row in tqdm(project_df.iter_rows(named=True), total=project_df.height):
            q_id = row.get("id", "")
            q_prompt = row.get("prompt", "")
            
            # The Magic: Ask the RAG agent in "full_text" mode!
            # We use k=5 to give Mistral enough context for complex questions
            try:
                answer = rag.ask(user_query=q_prompt, k=5, max_tokens=300, mode="full_text")
            except Exception as e:
                print(f"Fehler bei ID {q_id}: {e}")
                answer = "Fehler bei der Generierung."
                
            # Save the result to the CSV
            writer.writerow([q_id, answer])
            
    print(f"\n All done! The final answers are saved at: {PROJECT_OUTPUT_CSV}")